# Stage 16A: 6.685 / 6.693 frontier差分監査

CPU-onlyの静的監査です。230/240 Notebookのcode、入力dependency、最終hedge、sanitization contractを比較します。学習・提出は行いません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
artifact_dir = Path('/content/drive/MyDrive/kaggle/rogii/artifacts')
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
RUN_ID = 'stage16a_frontier_diff_v001'
run_dir = artifact_dir / RUN_ID
run_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    'uv', 'run', 'rogii-frontier-diff',
    '--config', 'configs/experiment/stage16a_frontier_diff.yaml',
    '--repo-dir', str(repo_dir), '--output-dir', str(run_dir),
], cwd=repo_dir, check=True)
report = json.loads((run_dir / 'frontier_diff.json').read_text())
{
    'winner': report['winner'],
    'score_delta_right_minus_left': report['score_delta_right_minus_left'],
    'changed_code_cells': report['changed_code_cells'],
    'upstream_identical': report['upstream_prediction_code_identical_through_cell_47'],
    'dependencies_identical': report['input_dependencies']['identical'],
    'sanitation_passed': all(report['sanitation_audit'].values()),
    'decision': report['decision'],
}

期待結果はwinner=`v599_a130_branch_conservative`、changed code cells=`[48, 50]`、upstream/dependencies/sanitationがすべてTrueです。これを確認後、Stage 16Bへ進みます。